# Clustering_PCA Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Grow some blobs.** Synthetic clusters come with ground truth for OUR evaluation — the algorithm receives only the coordinates.

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs

X, y_true = make_blobs(n_samples=150, centers=3, cluster_std=2.5,
                       random_state=42)

print("Shape:", X.shape)
print("True group sizes:", np.bincount(y_true).tolist())

# y_true exists only for grading afterwards - KMeans gets just X.

**2. Let K-Means loose.** `labels_` assigns every point a group; `cluster_centers_` are the groups' centres of gravity.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)

print("Found group sizes:", np.bincount(kmeans.labels_).tolist())
print("Centroids:\n", kmeans.cluster_centers_.round(2))

# random_state pins the starting centroids; n_init reruns the whole search
# 10 times and keeps the best - together they make results repeatable
# instead of dependent on one lucky start.

**3. Truth vs discovery.** Almost every row sits on the diagonal — but the labels themselves are arbitrary permutations.

In [ ]:
import pandas as pd

agreement = pd.crosstab(y_true, kmeans.labels_,
                        rownames=["true group"],
                        colnames=["found cluster"])
print(agreement)

# Cluster IDs are lottery tickets: refit and 0/1/2 can swap. Match groups
# by their MEMBERS and centroids, never by the numeric label.

## Part 2 — Practice

**4. Welcome, newcomers.** Prediction is nearest-centroid: `transform` even shows the distance to every cluster so you can see the decision.

In [ ]:
newcomers = pd.DataFrame({"visits": [2.0, 8.0], "spend": [8.0, 0.0]})
distances = pd.DataFrame(kmeans.transform(newcomers.to_numpy()).round(2),
                         columns=["dist_cluster_0", "dist_cluster_1",
                                  "dist_cluster_2"])
newcomers["assigned_cluster"] = kmeans.predict(newcomers.to_numpy())
print(newcomers.to_string(index=False))
print(distances.to_string(index=False))

# predict() measures the point's distance to every centroid and hands
# back the smallest - no votes, no magic.

**5. Hunt the elbow.** Inertia always falls as k grows; after the true cluster count each extra cluster barely dents it — that bend is your k.

In [ ]:
ks = range(1, 9)
inertias = [KMeans(n_clusters=k, n_init=10,
                   random_state=42).fit(X).inertia_ for k in ks]

prev = None
for k, ine in zip(ks, inertias):
    drop = "-" if prev is None else f"{prev - ine:,.0f}"
    print(f"k={k}: inertia={ine:>12,.0f}   drop vs previous: {drop}")
    prev = ine

# The plunge from k=2 to k=3 is dramatic; k=3 to k=4 barely moves. Extra
# clusters past the elbow split real groups instead of revealing new ones.

**6. Feed K-Means a banana.** One natural group goes in, three artificial wedges come out — spherical-equal-blob assumptions on full display.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

rng = np.random.default_rng(7)
t = rng.uniform(0, 10, 200)
banana = np.column_stack([t, 0.35 * rng.normal(size=200)])

km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(banana)
print("Slice sizes:", np.bincount(km.labels_).tolist())

# K-Means only understands "distance to a centre", so it carves the long
# thin group crosswise like a loaf. DBSCAN or Gaussian mixtures trace
# actual shapes instead.

**7. Shrink iris to 2-D.** Two rotated axes carry ~96% of the spread — but only because every column first stood on the same ruler.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X_iris = pd.DataFrame(iris.data, columns=iris.feature_names)

X_scaled = StandardScaler().fit_transform(X_iris)   # ALWAYS before PCA
pca = PCA(n_components=2).fit(X_scaled)

ratios = pca.explained_variance_ratio_
print("variance kept:", ratios.round(3), "-> total",
      round(ratios.sum(), 3))
print(pd.DataFrame(pca.components_, columns=X_iris.columns,
                   index=["PC1", "PC2"]).round(3))

# PCA maximises VARIANCE, and variance is unit-dependent: unscaled, the
# widest-measured column would become PC1 by brute force. Scaling puts
# all columns on one ruler so rotation tracks signal, not units.

## Part 3 — Challenge

**8. The 95% shortcut.** Pass a fraction instead of a count and sklearn picks the smallest set clearing the bar — compression with a quality guarantee.

In [ ]:
pca95 = PCA(n_components=0.95).fit(X_scaled)

kept = pca95.n_components_
total = pca95.explained_variance_ratio_.sum()
print(f"components chosen: {kept}")
print(f"variance retained: {total:.3f}")

# On dozens/hundreds of columns this auto-tunes dimensionality: models
# downstream train on fewer columns while you keep a written promise
# ("95% of the variance survived").

**9. Units hijack PCA.** Unscaled, the widest-ranged column quietly becomes PC1 — the rotation measured rulers, not structure.

In [ ]:
pca_raw = PCA(n_components=2).fit(X_iris)
pca_scaled = PCA(n_components=2).fit(X_scaled)

print(f"PC1 share, RAW features   : "
      f"{pca_raw.explained_variance_ratio_[0]:.3f}")
print(f"PC1 share, SCALED features: "
      f"{pca_scaled.explained_variance_ratio_[0]:.3f}")

top = abs(pca_raw.components_[0]).argmax()
print("raw PC1 is dominated by:", X_iris.columns[top])

# Whichever column happens to carry the biggest numbers seizes PC1 -
# information-free unit choice posing as insight. Scale first, always.